In [1]:
%%capture
!pip install peft
!pip install evaluate
!pip install datasets
!pip install "transformers==4.57.2"
!pip install sentencepiece
!pip install emoji

In [2]:
import os
import re
import shutil

import emoji
import evaluate
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

In [3]:
def clean_text(text):
    if pd.isna(text):
        return text

    # 1. lowercase
    text = text.lower()

    # 2. remove @USER mentions
    text = re.sub(r"@user", "", text, flags=re.IGNORECASE)
    text = re.sub(r"@url", "", text, flags=re.IGNORECASE)

    # 3. remove URLs (actual links or placeholder "URL")
    # text = re.sub(r'http\S+|https\S+|url', '', text, flags=re.IGNORECASE)

    # 4. remove underscores, repeated underscores
    text = re.sub(r"_+", " ", text)

    # 5. remove slashes
    text = text.replace("\\", " ").replace("/", " ")

    # 6. remove emojis
    text = emoji.replace_emoji(text, replace="")

    # 7. remove quotation marks (normal + smart)
    text = re.sub(r"[\"“”]", "", text)

    # 8. normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [4]:
# Setup
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [5]:
%%time
# Load Model & Tokenizer
MODEL_NAME = "jhu-clsp/mmBERT-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)

base_model.to(device)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.23G [00:00<?, ?B/s]

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at jhu-clsp/mmBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


CPU times: user 2.8 s, sys: 1.96 s, total: 4.75 s
Wall time: 6.44 s


ModernBertForSequenceClassification(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(256000, 768, padding_idx=0)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, out_features=768, bias=False)
        )
      )
 

model.safetensors:   0%|          | 0.00/1.23G [00:00<?, ?B/s]

In [6]:
# Load Data
DATA_DIR = "subtask1"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
DEV_DIR = os.path.join(DATA_DIR, "dev")
TEST_DIR = os.path.join(DATA_DIR, "test")


def load_split(split_dir):
    dfs = []
    if not os.path.exists(split_dir):
        print(f"Directory not found: {split_dir}")
        return pd.DataFrame()
    for file in os.listdir(split_dir):
        if file.endswith(".csv"):
            lang = file.replace(".csv", "")
            df = pd.read_csv(os.path.join(split_dir, file))
            df["lang"] = lang
            dfs.append(df)
    if not dfs:
        return pd.DataFrame()
    return pd.concat(dfs, ignore_index=True)


print("Loading Train Data...")
raw_train_df = load_split(TRAIN_DIR)
print(f"Loaded {len(raw_train_df)} training examples")

print("Loading Dev Data (Used as internal Test)...")
raw_dev_df = load_split(DEV_DIR)
print(f"Loaded {len(raw_dev_df)} dev examples")

print("Loading Test Data (For Submission)...")
raw_test_df = load_split(TEST_DIR)
print(f"Loaded {len(raw_test_df)} test examples")

Loading Train Data...
Loaded 73681 training examples
Loading Dev Data (Used as internal Test)...
Loaded 3687 dev examples
Loading Test Data (For Submission)...
Loaded 33288 test examples


In [7]:
# Preprocess Text
print("Preprocessing text (cleaning)...")
raw_train_df["text"] = raw_train_df["text"].astype(str).apply(clean_text)
raw_dev_df["text"] = raw_dev_df["text"].astype(str).apply(clean_text)
raw_test_df["text"] = raw_test_df["text"].astype(str).apply(clean_text)

Preprocessing text (cleaning)...


In [8]:
# Data Splitting
# Rename 'polarization' to 'labels'
if "polarization" in raw_train_df.columns:
    raw_train_df = raw_train_df.rename(columns={"polarization": "labels"})
if "polarization" in raw_dev_df.columns:
    raw_dev_df = raw_dev_df.rename(columns={"polarization": "labels"})

# Split Train into 95% Train / 5% Val
train_df, val_df = train_test_split(
    raw_train_df,
    test_size=0.05,
    stratify=raw_train_df["labels"],
    random_state=SEED,
    shuffle=True,
)

# Use Dev as Internal Test
test_df = raw_dev_df.copy()

print("Shape after split:")
print(f"Train:      {train_df.shape}")
print(f"Validation: {val_df.shape}")
print(f"Test (Dev): {test_df.shape}")

Shape after split:
Train:      (69996, 4)
Validation: (3685, 4)
Test (Dev): (3687, 4)


In [9]:
# Create Dataset Objects
train_dataset = Dataset.from_pandas(train_df[["text", "labels"]], preserve_index=False)
val_dataset = Dataset.from_pandas(val_df[["text", "labels"]], preserve_index=False)
test_dataset = Dataset.from_pandas(test_df[["text", "labels"]], preserve_index=False)

dataset = DatasetDict(
    {"train": train_dataset, "validation": val_dataset, "test": test_dataset}
)

In [10]:
%%time


# Tokenize
def tokenize_function(examples):
    return tokenizer(
        examples["text"], padding="max_length", truncation=True, max_length=256
    )


print("Tokenizing datasets...")
encoded_dataset = dataset.map(tokenize_function, batched=True)
encoded_dataset.set_format(
    type="torch", columns=["input_ids", "attention_mask", "labels"]
)

Tokenizing datasets...


Map:   0%|          | 0/69996 [00:00<?, ? examples/s]

Map:   0%|          | 0/3685 [00:00<?, ? examples/s]

Map:   0%|          | 0/3687 [00:00<?, ? examples/s]

CPU times: user 9.58 s, sys: 278 ms, total: 9.85 s
Wall time: 5.67 s


In [11]:
# Training Arguments
OUTPUT_DIR = "./output_results"
BATCH_SIZE = 32
EPOCHS = 30
LR = 2e-5
GRAD_ACCUM = 2

# Calculate steps
steps_per_epoch = len(encoded_dataset["train"]) // (BATCH_SIZE * GRAD_ACCUM)
eval_steps = steps_per_epoch
steps_per_epoch

1093

In [12]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_steps=1000,
    gradient_accumulation_steps=GRAD_ACCUM,
    logging_steps=eval_steps,
    eval_steps=eval_steps,
    save_steps=eval_steps * 10,  # Save less frequently to save space
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    eval_strategy="steps",
    logging_dir=f"{OUTPUT_DIR}/logs",
    report_to="none",
    fp16=torch.cuda.is_available(),
)

metric_f1 = evaluate.load("f1")
metric_acc = evaluate.load("accuracy")


def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="macro")[
        "f1"
    ]
    acc = metric_acc.compute(predictions=predictions, references=labels)["accuracy"]
    return {"f1": f1, "accuracy": acc}


trainer = Trainer(
    model=base_model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

In [13]:
%%time
# Start Training
trainer.train()

Step,Training Loss,Validation Loss,F1,Accuracy
1093,1.052800,0.423423,0.797564,0.798643
2186,0.740800,0.406571,0.819463,0.821167
3279,0.416100,0.522809,0.793758,0.797286
4372,0.165100,0.787736,0.807188,0.807598
5465,0.112600,1.335560,0.801302,0.801357


CPU times: user 1h 8min 30s, sys: 8.75 s, total: 1h 8min 39s
Wall time: 18min 49s


TrainOutput(global_step=5465, training_loss=0.49749424437918144, metrics={'train_runtime': 1128.9032, 'train_samples_per_second': 1860.106, 'train_steps_per_second': 29.072, 'total_flos': 5.957810179743744e+16, 'train_loss': 0.49749424437918144, 'epoch': 4.995429616087751})

In [14]:
# Evaluation on Internal Test Set (Dev Folder)
print("Evaluating on Internal Test Set (Dev folder data)...")
preds_output = trainer.predict(encoded_dataset["test"])

pred_labels = np.argmax(preds_output.predictions, axis=1)
true_labels = preds_output.label_ids

print("\nClassification Report:")
report = classification_report(
    true_labels, pred_labels, target_names=["Not Polar (0)", "Polar (1)"], digits=4
)
print(f"\n{report}")

macro_f1 = f1_score(true_labels, pred_labels, average="macro")
print(f"Macro F1: {macro_f1:.4f}")

# Per-Language Analysis
test_df["preds"] = pred_labels
print("\n=== Macro F1 per Language ===")
results = []
for lang in sorted(test_df["lang"].unique()):
    lang_df = test_df[test_df["lang"] == lang]
    f1 = f1_score(lang_df["labels"], lang_df["preds"], average="macro")
    acc = accuracy_score(lang_df["labels"], lang_df["preds"])
    print(f"{lang}: F1={f1:.4f}, Acc={acc:.4f}, Support={len(lang_df)}")
    results.append(
        {"lang": lang, "f1_macro": f1, "accuracy": acc, "count": len(lang_df)}
    )

results_df = pd.DataFrame(results)
print(f"\nAverage Macro F1 across languages: {results_df['f1_macro'].mean():.4f}")

Evaluating on Internal Test Set (Dev folder data)...



Classification Report:

               precision    recall  f1-score   support

Not Polar (0)     0.7572    0.8245    0.7895      1744
    Polar (1)     0.8289    0.7627    0.7944      1943

     accuracy                         0.7920      3687
    macro avg     0.7930    0.7936    0.7919      3687
 weighted avg     0.7950    0.7920    0.7921      3687

Macro F1: 0.7919

=== Macro F1 per Language ===
amh: F1=0.6703, Acc=0.7651, Support=166
arb: F1=0.7836, Acc=0.7870, Support=169
ben: F1=0.7998, Acc=0.8133, Support=166
deu: F1=0.7006, Acc=0.7044, Support=159
eng: F1=0.7266, Acc=0.7625, Support=160
fas: F1=0.8592, Acc=0.8841, Support=164
hau: F1=0.6055, Acc=0.9066, Support=182
hin: F1=0.7474, Acc=0.8540, Support=137
ita: F1=0.6498, Acc=0.6506, Support=166
khm: F1=0.6490, Acc=0.8916, Support=332
mya: F1=0.8055, Acc=0.8056, Support=144
nep: F1=0.8700, Acc=0.8700, Support=100
ori: F1=0.6623, Acc=0.7712, Support=118
pan: F1=0.7050, Acc=0.7100, Support=100
pol: F1=0.7011, Acc=0.7143, Suppor

In [15]:
# Generate Submission (Test Folder)
SUBMISSION_DIR = "./subtask_1"
if os.path.exists(SUBMISSION_DIR):
    shutil.rmtree(SUBMISSION_DIR)
os.makedirs(SUBMISSION_DIR)

print("Generating predictions for submission...")

# tokenize test set for submission
submission_dataset = Dataset.from_pandas(raw_test_df[["text"]], preserve_index=False)
submission_tokenized = submission_dataset.map(tokenize_function, batched=True)
submission_tokenized.set_format(type="torch", columns=["input_ids", "attention_mask"])

# Predict
submission_preds_output = trainer.predict(submission_tokenized)
submission_labels = np.argmax(submission_preds_output.predictions, axis=1)

# Add predictions back to dataframe
raw_test_df["polarization"] = submission_labels

# Save individual files
languages = sorted(raw_test_df["lang"].unique())
print(f"Processing {len(languages)} languages for submission...")

for lang in languages:
    lang_df = raw_test_df[raw_test_df["lang"] == lang]
    output_df = lang_df[["id", "polarization"]]

    output_path = os.path.join(SUBMISSION_DIR, f"pred_{lang}.csv")
    output_df.to_csv(output_path, index=False)

print("Zipping prediction files...")
shutil.make_archive("subtask_1", "zip", SUBMISSION_DIR)
print(f"Created subtask_1.zip in {os.getcwd()}")

Generating predictions for submission...


Map:   0%|          | 0/33288 [00:00<?, ? examples/s]

Processing 22 languages for submission...
Zipping prediction files...
Created subtask_1.zip in /home/jovyan/work
